In [1]:
import torch
import torch.nn as nn

In [4]:
def BCELoss(y_hat, y_true):
    y_hat = torch.clamp(y_hat, 1e-7, 1 - 1e-7)
    loss = -torch.mean(y_true * torch.log(y_hat) + (1 - y_true) * torch.log(1 - y_hat))
    return loss

In [5]:
class ReLU:
    def __init__(self):
        pass
        
    def forward(self, z):
        # We can use PyTorch's built-in, or write it from scratch using clamping
        return torch.maximum(z, torch.tensor(0.0))
        
    def backward(self, lr=None):
        # No weights to update, so we do nothing!
        pass

class Flatten:
    def __init__(self):
        pass
        
    def forward(self, x):
        batch_size = x.shape[0]
        return x.view(batch_size, -1)
        
    def backward(self, lr=None):
        pass

In [7]:
class Dense:
    def __init__(self, in_features, out_features):
        # Initialize weights with requires_grad=True
        # Using a simple scaling for initialization
        self.W = torch.randn(in_features, out_features, requires_grad=True)
        self.b = torch.zeros(out_features, requires_grad=True)
        
        # Scale weights appropriately to avoid exploding gradients
        with torch.no_grad():
            self.W *= 0.01 

    def forward(self, X):
        # PyTorch records this matrix multiplication and addition
        return X @ self.W + self.b
        
    def backward(self, lr):
        # PyTorch has already computed self.W.grad and self.b.grad!
        # We temporarily disable gradient tracking to update the weights
        with torch.no_grad():
            self.W -= lr * self.W.grad
            self.b -= lr * self.b.grad
            
            # CRITICAL: Clear the gradients for the next training step
            self.W.grad.zero_()
            self.b.grad.zero_()

In [13]:
class Conv2d:
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        
        # Initialize weights and biases
        # We use a standard uniform initialization limit based on kernel size
        limit = (kernel_size * kernel_size * in_channels) ** -0.5
        self.W = torch.FloatTensor(out_channels, in_channels, kernel_size, kernel_size).uniform_(-limit, limit)
        self.W.requires_grad = True
        
        self.b = torch.zeros(out_channels, requires_grad=True)

    def forward(self, X):
        batch_size, _, h_in, w_in = X.shape
        
        # Calculate output dimensions
        h_out = (h_in + 2 * self.padding - self.kernel_size) // self.stride + 1
        w_out = (w_in + 2 * self.padding - self.kernel_size) // self.stride + 1
        
        # Apply padding if needed (using PyTorch's padding utility)
        if self.padding > 0:
            import torch.nn.functional as F
            X_padded = F.pad(X, (self.padding, self.padding, self.padding, self.padding))
        else:
            X_padded = X
            
        out_pixels = []
        
        # The sliding window
        for i in range(h_out):
            for j in range(w_out):
                h_start = i * self.stride
                h_end = h_start + self.kernel_size
                w_start = j * self.stride
                w_end = w_start + self.kernel_size
                
                # Extract the region: (batch_size, in_channels, kernel_size, kernel_size)
                region = X_padded[:, :, h_start:h_end, w_start:w_end]
                
                # Element-wise multiplication, sum, and bias addition
                # region.unsqueeze(1) broadcasts the shape to match the weights correctly
                pixel_val = torch.sum(region.unsqueeze(1) * self.W, dim=(2, 3, 4)) + self.b
                out_pixels.append(pixel_val)
                
        # Stack the calculated pixels and reshape back into an image tensor
        out = torch.stack(out_pixels, dim=-1).view(batch_size, self.out_channels, h_out, w_out)
        return out

    def backward(self, lr):
        # Update parameters and clear the tape recorder!
        with torch.no_grad():
            if self.W.grad is not None:
                self.W -= lr * self.W.grad
                self.W.grad.zero_()
            if self.b.grad is not None:
                self.b -= lr * self.b.grad
                self.b.grad.zero_()

In [10]:
class MaxPool:
    def __init__(self, pool_size=2, stride=2):
        self.pool_size = pool_size
        self.stride = stride

    def forward(self, X):
        batch_size, channels, h_in, w_in = X.shape
        
        # Calculate output dimensions
        h_out = (h_in - self.pool_size) // self.stride + 1
        w_out = (w_in - self.pool_size) // self.stride + 1
        
        out_pixels = []
        
        for i in range(h_out):
            for j in range(w_out):
                h_start = i * self.stride
                h_end = h_start + self.pool_size
                w_start = j * self.stride
                w_end = w_start + self.pool_size
                
                # Extract the region
                region = X[:, :, h_start:h_end, w_start:w_end]
                
                # Find the maximum value over the spatial dimensions (height and width)
                # dim=(2,3) computes the max across the height and width of the patch
                max_val = torch.amax(region, dim=(2, 3))
                out_pixels.append(max_val)
                
        # Stack and reshape
        out = torch.stack(out_pixels, dim=-1).view(batch_size, channels, h_out, w_out)
        return out

    def backward(self, lr=None):
        # No weights to update!
        pass

In [8]:
class CNN:
    def __init__(self, layers):
        self.layers = layers
        
    def forward(self, X):
        out = X
        for layer in self.layers:
            out = layer.forward(out)
        return out

    def backward(self, lr):
        # We no longer pass dL backwards manually.
        # We just tell each layer to update its weights using the learning rate.
        for layer in self.layers:
            layer.backward(lr)

In [14]:
# Example instantiation
model = CNN([
    Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1),
    ReLU(),
    MaxPool(pool_size=2, stride=2),
    Flatten(),
    Dense(in_features=8 * 14 * 14, out_features=1) # Assuming a 28x28 input image
])

In [12]:
model